In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from loaders._load_vn30_binary import preprocess, VN30, TARGETS
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier


In [10]:
acc = []
for symbol in VN30:
    data = preprocess(symbol, lag=30)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]

    tscv = TimeSeriesSplit(n_splits=5)

In [11]:
def knn_pro_for_symbol(symbol: str, lag: int = 30):
    """Huấn luyện KNN (classification) cho 1 mã, trả về dict kết quả."""
    data = preprocess(symbol, lag=lag, use_rolling=True, use_calendar=True, feat_select=True)
    X_train, y_train = data["train"]
    X_test,  y_test  = data["test"]

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(svd_solver="full")),
        ("knn", KNeighborsClassifier())
    ])

    param_grid = {
        "pca__n_components": [None, 0.90, 0.95],
        "knn__n_neighbors": [3, 5, 7, 11, 15],
        "knn__weights": ["uniform", "distance"],
        "knn__p": [1, 2],
    }

    cv = TimeSeriesSplit(n_splits=5)
    gs = GridSearchCV(pipe, param_grid, cv=cv,
                      scoring="balanced_accuracy",
                      n_jobs=1,      
                      refit=True, verbose=0)
    gs.fit(X_train, y_train)

    y_pred = gs.predict(X_test)
    test_ba = balanced_accuracy_score(y_test, y_pred)

    return {
        "symbol": symbol,
        "best_params": gs.best_params_,
        "cv_best_ba": gs.best_score_,
        "test_ba": test_ba,
    }


all_results = []
for sym in VN30:
    res = knn_pro_for_symbol(sym, lag=30)
    print(f"[{sym}] test BA={res['test_ba']:.3f} | cv BA={res['cv_best_ba']:.3f} | {res['best_params']}")
    all_results.append(res)

df = pd.DataFrame(all_results).sort_values("test_ba", ascending=False).reset_index(drop=True)

top_k = 10
top10 = df.head(top_k)

top10_mean = top10["test_ba"].mean()
top10_std  = top10["test_ba"].std(ddof=1)   # std mẫu

print("\n=== Top 10 symbols by test Balanced Accuracy ===")
print(top10[["symbol", "test_ba", "cv_best_ba", "best_params"]])

print(f"\nTop-10 mean BA: {top10_mean:.3f}")
print(f"Top-10 std  BA: {top10_std:.3f}")




[ACB] test BA=0.494 | cv BA=0.515 | {'knn__n_neighbors': 7, 'knn__p': 2, 'knn__weights': 'uniform', 'pca__n_components': 0.9}
[BCM] test BA=0.532 | cv BA=0.526 | {'knn__n_neighbors': 7, 'knn__p': 2, 'knn__weights': 'uniform', 'pca__n_components': 0.9}
[BID] test BA=0.480 | cv BA=0.517 | {'knn__n_neighbors': 7, 'knn__p': 2, 'knn__weights': 'uniform', 'pca__n_components': 0.95}
[BVH] test BA=0.562 | cv BA=0.530 | {'knn__n_neighbors': 15, 'knn__p': 1, 'knn__weights': 'uniform', 'pca__n_components': None}
[CTG] test BA=0.490 | cv BA=0.511 | {'knn__n_neighbors': 15, 'knn__p': 2, 'knn__weights': 'uniform', 'pca__n_components': 0.9}
[FPT] test BA=0.532 | cv BA=0.530 | {'knn__n_neighbors': 11, 'knn__p': 2, 'knn__weights': 'uniform', 'pca__n_components': None}
[GAS] test BA=0.517 | cv BA=0.499 | {'knn__n_neighbors': 11, 'knn__p': 1, 'knn__weights': 'uniform', 'pca__n_components': 0.95}
[GVR] test BA=0.512 | cv BA=0.507 | {'knn__n_neighbors': 7, 'knn__p': 1, 'knn__weights': 'uniform', 'pca__n_co